# Process Mining Foundations

**Overview:** Process mining is the art and science of discovering, analyzing, and improving business processes through the systematic analysis of event logs.

## What You'll Learn
1. **Event Log Structure** – Data format for process mining
2. **Petri Nets** – Formal process models with visual representation
3. **Causal Nets** – Simplified causality models
4. **Workflow Nets** – Structured process patterns
5. **Alpha Algorithm** – Automated process discovery
6. **Conformance** – Comparing processes to reality

<a href="https://colab.research.google.com/github/solutiongate-learn/bizlens/blob/main/notebooks/Process_Mining_Foundations.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
# Setup: BizLens in Google Colab
# This cell installs dependencies and prepares the environment

# Install BizLens
!pip install bizlens networkx -q

# Check if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False

# Test import
import bizlens as bl
print("✓ BizLens loaded successfully")

## 1. Introduction to Process Mining

Process mining bridges the gap between **business process management (BPM)** and **data mining**.

### Three Main Types:
- **Process Discovery** – Extract process models from event logs
- **Conformance Checking** – Compare recorded processes against a model
- **Performance Analysis** – Identify bottlenecks and delays

### Real-World Applications:
- Manufacturing: Identify production bottlenecks
- Healthcare: Improve patient pathways
- Finance: Detect fraudulent transactions
- IT: Optimize incident resolution workflows

## 2. Event Log Structure

An **event log** contains:

| Field | Description | Example |
|-------|-------------|---------|
| **Case ID** | Process instance identifier | `Order_12345` |
| **Activity** | Action performed | `Submit Order` |
| **Timestamp** | When it happened | `2024-04-09 14:30` |
| **Resource** | Who performed it | `Alice` |
| **Attributes** | Additional data | Amount: $100 |

**Key Properties:**
- One log = multiple cases (process instances)
- Each case = ordered events (activities)
- Events ordered by timestamp

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Create sample event log
np.random.seed(42)
cases = [f'Order_{i}' for i in range(1, 101)]
activities = ['Submit Order', 'Payment', 'Pick Items', 'Pack', 'Ship', 'Deliver', 'Feedback']
resources = ['Alice', 'Bob', 'System', 'Warehouse']

events = []
base_time = datetime(2024, 1, 1)

for case_id in cases:
    current_time = base_time + timedelta(days=np.random.randint(0, 90))
    for activity in activities:
        if np.random.random() > 0.1:
            events.append({
                'case_id': case_id,
                'activity': activity,
                'timestamp': current_time,
                'resource': np.random.choice(resources),
                'duration_minutes': np.random.randint(5, 120)
            })
            current_time += timedelta(hours=np.random.randint(1, 24))

event_log = pd.DataFrame(events).sort_values(['case_id', 'timestamp']).reset_index(drop=True)

print(f"Event Log Summary:")
print(f"  Total Events: {len(event_log)}")
print(f"  Unique Cases: {event_log['case_id'].nunique()}")
print(f"  Unique Activities: {event_log['activity'].nunique()}")
print(f"\nFirst 10 Events:")
print(event_log.head(10))

## 3. Petri Nets

A **Petri Net** is a formal graphical model for representing processes.

### Components:
- **Places (○)** – States/conditions (blue circles)
- **Transitions (▮)** – Activities/events (red rectangles)
- **Arcs (→)** – Flow connections
- **Tokens (●)** – Mobile objects (process instances)

### Execution Model:
1. Start with token in initial place
2. Transition fires when all inputs have tokens
3. Firing removes tokens from inputs, adds to outputs
4. Process ends when token reaches final place

### Properties:
- **Soundness** – No deadlocks
- **Liveness** – All transitions eventually fireable
- **Safeness** – Max one token per place

In [ ]:
import bizlens as bl

# Generate Petri net from event log
petri_net = bl.process_mining.petri_net_from_log(
    event_log,
    case_id_col='case_id',
    activity_col='activity'
)

print("Petri Net Structure:")
print(f"  Places: {len(petri_net['places'])}")
print(f"  Transitions: {len(petri_net['transitions'])}")
print(f"  Arcs: {len(petri_net['arcs'])}")
print(f"  Start: {petri_net['initial_marking']}")
print(f"  End: {petri_net['final_marking']}")

In [ ]:
# Visualize Petri Net
fig = bl.process_mining.visualize_petri_net(
    petri_net,
    title="Petri Net: Order Processing"
)
plt.tight_layout()
plt.show()

print("Legend: ○ = Places | ▮ = Transitions | → = Arcs")

## 4. Causal Nets

**Causal Nets (C-Nets)** show direct causality between activities without explicit places.

### Advantages:
- Simpler visual representation
- Easier for stakeholders to understand
- Handles concurrency naturally
- Focus on activity relationships

### Example:
```
Submit Order → Payment → Pick Items → Pack → Ship → Deliver
```

In [ ]:
# Discover causal net
causal_net = bl.process_mining.causal_net_from_log(
    event_log,
    case_id_col='case_id',
    activity_col='activity',
    min_support=0.05
)

print(f"Causal Net Discovery:")
print(f"  Activities: {len(causal_net['activities'])}")
print(f"  Causal Relations: {len(causal_net['causal_edges'])}")
print(f"  Cases Analyzed: {causal_net['total_cases']}")
print(f"\nTop Causal Edges:")
sorted_edges = sorted(causal_net['causal_edges'].items(), 
                      key=lambda x: x[1], reverse=True)[:5]
for (from_act, to_act), count in sorted_edges:
    print(f"  {from_act} → {to_act} (count: {count})")

In [ ]:
# Visualize causal net
fig = bl.process_mining.visualize_causal_net(
    causal_net,
    title="Causal Net: Activity Relationships"
)
fig.show()

## 5. Alpha Algorithm

**Alpha Algorithm** automatically discovers process models from event logs.

### Steps:
1. Extract all activities from the log
2. Find direct succession pairs (A followed by B)
3. Identify causality (A→B if A>B and B>A doesn't happen)
4. Find start and end activities
5. Build Petri net from these relations

### Limitations:
- Cannot handle loops of length ≤2
- Struggles with implicit concurrency
- May produce overly general "flower" models
- Better for sequential, structured processes

In [ ]:
# Apply Alpha Algorithm
alpha_result = bl.process_mining.alpha_algorithm(
    event_log,
    case_id_col='case_id',
    activity_col='activity'
)

print("Alpha Algorithm Results:")
print(f"  Activities: {len(alpha_result['activities'])}")
print(f"  Direct Succession: {len(alpha_result['direct_succession'])} pairs")
print(f"  Causality Relations: {len(alpha_result['causality_relations'])} relations")
print(f"\nStart Activities: {alpha_result['start_activities']}")
print(f"End Activities: {alpha_result['end_activities']}")
print(f"\nFirst 5 Causality Relations:")
for i, pair in enumerate(sorted(alpha_result['causality_relations'])[:5]):
    print(f"  {pair[0]} → {pair[1]}")

## 6. Workflow Nets

**Workflow Nets (WF-Nets)** are well-structured Petri nets with:

### Requirements:
- Single source place (initial state)
- Single sink place (final state)
- All nodes reachable from source to sink
- No deadlocks or unbounded behaviors

### Why Important:
- Ensures valid process execution
- Guarantees termination
- Enables automated analysis
- Foundation for process mining algorithms

In [ ]:
# Validate workflow net
def validate_workflow_net(petri_net):
    """Check if Petri net is a valid Workflow Net."""
    initial = petri_net['initial_marking']
    final = petri_net['final_marking']
    
    is_valid = len(initial) == 1 and len(final) == 1
    
    return {
        'is_workflow_net': is_valid,
        'has_single_source': len(initial) == 1,
        'has_single_sink': len(final) == 1,
        'source_place': list(initial.keys())[0] if initial else None,
        'sink_place': list(final.keys())[0] if final else None
    }

validation = validate_workflow_net(petri_net)
print("Workflow Net Validation:")
for k, v in validation.items():
    print(f"  {k}: {v}")

## 7. Conformance Checking

**Conformance Checking** validates: "Does actual execution match our process model?"

### Key Metrics:
- **Fitness** – % of cases that replay correctly (log fits model)
- **Precision** – How specific is the model? (doesn't allow too much)
- **Generalization** – Can model handle unseen variants?
- **Simplicity** – Is the model as simple as possible?

### Token Replay Algorithm:
1. Start with initial marking
2. For each event, fire the transition
3. If transition can't fire → missing token (fitness issue)
4. If different token type can fire → precision issue

In [ ]:
# Simple fitness calculation
def calculate_fitness(event_log, petri_net, 
                     case_id_col='case_id', activity_col='activity'):
    """Calculate fitness: % of cases that can be fully replayed."""
    replayed = 0
    total = 0
    
    for case_id in event_log[case_id_col].unique():
        total += 1
        case_events = event_log[event_log[case_id_col] == case_id]
        activities = case_events[activity_col].tolist()
        
        # Check if all activities in model
        all_valid = all(act in petri_net['transitions'] for act in activities)
        if all_valid:
            replayed += 1
    
    fitness = (replayed / total * 100) if total > 0 else 0
    return fitness, replayed, total

fitness, replayed, total = calculate_fitness(event_log, petri_net)
print(f"Conformance Checking - Fitness:")
print(f"  Cases Replayed: {replayed}/{total}")
print(f"  Fitness Score: {fitness:.1f}%")
print(f"\nInterpretation:")
print(f"  {fitness:.1f}% of cases follow the discovered model exactly")

## Summary & Next Steps

### Concepts Covered:
✓ Event log structure  
✓ Petri nets for process modeling  
✓ Causal nets for simplified models  
✓ Workflow nets for well-formed processes  
✓ Alpha algorithm for automated discovery  
✓ Conformance checking for validation  

### Practical Workflow:
1. **Collect** event logs from your system
2. **Explore** with transitions and timelines
3. **Discover** process models (Alpha algorithm)
4. **Visualize** as Petri nets or causal nets
5. **Validate** with conformance checking
6. **Optimize** by finding bottlenecks

### Advanced Topics (Future):
- Heuristic Miner (handles complex processes)
- Inductive Mining (better concurrency handling)
- Temporal analysis (speed improvements)
- Variant discovery (process deviations)
- Streaming analysis (real-time monitoring)